In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Googleドライブ内にモデル保存用フォルダを作成して設定
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'
print("✅ Googleドライブを接続しました！モデルはドライブに永続保存されます。")

Mounted at /content/drive
✅ Googleドライブを接続しました！モデルはドライブに永続保存されます。


In [ ]:
import os
import subprocess
import time
import shutil

# 破損ファイルや過去のゴミをクリーンアップ
for path in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/content/cloudflared.deb"]:
    if os.path.exists(path):
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            else:
                os.remove(path)
        except Exception:
            pass

def download_file(url, filepath):
    try:
        subprocess.run(["curl", "-L", "-o", filepath, url], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

# 0. 依存パッケージ (zstd + L4 GPU自動検出用pciutils) をインストール
print("📦 依存パッケージ (zstd, pciutils) をインストール中...")
subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "zstd", "pciutils"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 1. Ollama のインストール (L4のAda Lovelaceアーキテクチャを自動認識)
print("📦 Ollama をインストール中...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. cloudflared のインストール
print("📦 Cloudflare Tunnel を準備中...")
cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
success_cf = download_file(cf_url, "/content/cloudflared.deb")

if success_cf and os.path.exists("/content/cloudflared.deb") and os.path.getsize("/content/cloudflared.deb") > 5000000:
    !dpkg -i /content/cloudflared.deb > /dev/null 2>&1
    print("✅ Cloudflare Tunnel の準備完了")
else:
    raise RuntimeError("❌ Cloudflare Tunnel のダウンロードに失敗しました。")

# 3. Ollama サーバーをバックグラウンドで起動
print("🚀 Ollama サーバーを起動中...")
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
if os.path.exists('/content/drive/MyDrive'):
    os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# 4. L4 GPU 最適化モデル 2種をセットアップ
MODELS = [
    "qwen2.5-coder:14b",  # ★超高知能コーディングモデル（L4の真骨頂）
    "codegemma:7b"        # ★Google DeepMind製コーディングモデル
]

for model in MODELS:
    print(f"📥 モデル ({model}) を準備中...")
    !ollama pull {model}

# 5. Cloudflare Tunnel を起動してURLを発行
print("\n🌐 Cloudflare Tunnel を起動中...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

for line in tunnel.stderr:
    if "trycloudflare.com" in line and "https://" in line:
        for part in line.split():
            if "trycloudflare.com" in part and part.startswith("https://"):
                url = part.strip()
                print("\n" + "="*60)
                print("🎉 準備完了！以下のURLを教えてください：")
                print(f"👉 {url}")
                print("="*60 + "\n")
                break
        break